In [2]:
app_code = """
from flask import Flask, render_template, request, redirect, session
from flask_mysqldb import MySQL
import MySQLdb.cursors
from preprocess import clean_text
from lstm_predict import predict_lstm
import config

app = Flask(__name__)
app.secret_key = 'your_secret_key'

app.config['MYSQL_HOST'] = config.MYSQL_HOST
app.config['MYSQL_USER'] = config.MYSQL_USER
app.config['MYSQL_PASSWORD'] = config.MYSQL_PASSWORD
app.config['MYSQL_DB'] = config.MYSQL_DB

mysql = MySQL(app)

@app.route('/')
def home():
    return redirect('/login')

@app.route('/about')
def about():
    return render_template('about.html')

@app.route('/login', methods=['GET', 'POST'])
def login():
    msg = ''
    if request.method == 'POST':
        username = request.form['username']
        password = request.form['password']
        cursor = mysql.connection.cursor(MySQLdb.cursors.DictCursor)
        cursor.execute('SELECT * FROM users WHERE username = %s AND password = %s', (username, password,))
        account = cursor.fetchone()
        if account:
            session['loggedin'] = True
            session['id'] = account['id']
            session['username'] = account['username']
            return redirect('/input')
        else:
            msg = 'Incorrect username/password!'
    return render_template('login.html', msg=msg)

@app.route('/signup', methods=['GET', 'POST'])
def signup():
    msg = ''
    if request.method == 'POST':
        username = request.form['username']
        email = request.form['email']
        password = request.form['password']
        cursor = mysql.connection.cursor(MySQLdb.cursors.DictCursor)
        cursor.execute('SELECT * FROM users WHERE username = %s', (username,))
        account = cursor.fetchone()
        if account:
            msg = 'Account already exists!'
        else:
            cursor.execute('INSERT INTO users (username, email, password) VALUES (%s, %s, %s)', (username, email, password))
            mysql.connection.commit()
            msg = 'You have successfully registered!'
    return render_template('signup.html', msg=msg)

@app.route('/forget', methods=['GET', 'POST'])
def forget():
    msg = ''
    if request.method == 'POST':
        email = request.form['email']
        cursor = mysql.connection.cursor(MySQLdb.cursors.DictCursor)
        cursor.execute('SELECT password FROM users WHERE email = %s', (email,))
        account = cursor.fetchone()
        if account:
            msg = f"Your password is: {account['password']}"
        else:
            msg = 'Email not found!'
    return render_template('forget.html', msg=msg)

@app.route('/input', methods=['GET', 'POST'])
def input_page():
    if 'loggedin' in session:
        if request.method == 'POST':
            tweet = request.form['tweet']
            cleaned = clean_text(tweet)
            prediction = predict_lstm(cleaned)
            cursor = mysql.connection.cursor(MySQLdb.cursors.DictCursor)
            cursor.execute('INSERT INTO inputs (user_id, tweet, sentiment) VALUES (%s, %s, %s)', 
                           (session['id'], tweet, prediction))
            mysql.connection.commit()
            return render_template('result.html', tweet=tweet, sentiment=prediction)
        return render_template('input.html')
    return redirect('/login')

@app.route('/logout')
def logout():
    session.clear()
    return redirect('/login')

if __name__ == '__main__':
    app.run(debug=True)
"""
with open("app.py", "w") as f:
    f.write(app_code)


In [3]:
!flask --app app.py run

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\SPURGE\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
2025-07-14 12:45:58.736312: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-14 12:46:03.904434: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
Traceback (most recent call last):
  File "C:\Users\SPURGE\anaconda3\Scripts\flask-script.py", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "C:\Users\SPURGE\anaconda3\Lib\site-packages\flask\cli.py", line 1105, in main
    cli.